# Nepal Generation Expansion Planning — 2030
## Complete, reproducible, beginner-readable optimization notebook

**Output:** optimal *new* capacity (MW), total 2030 capacity (MW), annual electricity mix (GWh and %), costs, model checks, and dry/wet daily dispatch plots.

**Built from your `1.ipynb` and Steps 1–40.** This is a clean **run-from-top replacement**: your original notebook's PDF-parsing cells depended on missing intermediates, so those fragile cells have been replaced by an explicitly sourced, reconciling data ledger. This workbook **does not assert that unverified hydropower is classified**.

**Study boundary:** one-node Nepal domestic grid, calendar 2030, JICA middle-case domestic demand adjusted for losses, 12 representative 24-hour days (288 optimization hours) weighted by actual days/month. Since every day in a month has the *same assumed input shape*, the 288-hour solution can be expanded to an equivalent 8,760-hour modeled dispatch for this simplified case.

**Technology decisions:** ROR, PROR, reservoir/storage hydropower, solar PV; imports and deterministic unserved demand. **BESS is excluded**, as requested. Existing Kulekhani chain is kept separate from new seasonal storage. This is a *scenario-based model*, **not** a verified prediction of what Nepal will build.

> Do not publish the 2030 optimum as a verified national construction requirement until remaining IPP classifications, commissioning schedules, solar-resource observations and new-project financing/cost evidence are resolved. Values labelled **proxy / assumption** below are intentionally editable and visible.

### Workflow — read from top to bottom
1. Environment and files → 2. audited 2026 capacity ledger → 3. 2030 demand → 4. hydro and solar resources → 5. costs and constraints → 6. build and solve LP → 7. results/plots → 8. full-year reconstruction and validation.

**No PuLP imports:** this notebook uses **SciPy HiGHS**, available in normal Google Colab. This avoids the repeated PuLP import errors in earlier versions. A short, well-commented model-builder cell organizes the linear constraints; the equations themselves remain visible.

## 1. Prepare Google Colab and output folders
Run this cell. It tries your existing `GEP_ERC/Inputs` and `GEP_ERC/Cleaned_Inputs` on Google Drive. All *core* results can still be reproduced without those folders, using the transparent saved aggregates and calculations from the earlier verified steps.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from scipy.sparse import coo_matrix

# Mounting is optional when running outside Google Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local Python mode: Google Drive is not needed.')

home = Path('/content/drive/MyDrive/GEP_ERC')
if home.exists():
    input_folder = home / 'Inputs'
    clean_folder = home / 'Cleaned_Inputs'
    output_folder = home / 'Model_2030_Outputs'
else:
    input_folder = Path('/mnt/data') if Path('/mnt/data').exists() else Path('/content')
    clean_folder = input_folder
    output_folder = input_folder / 'GEP_2030_Outputs'

output_folder.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 40)
print('Reference data folder:', clean_folder)
print('Outputs will be saved in:', output_folder)
print('SciPy linear-programming solver ready.')

## 2. Audit existing 2026 installed capacity (NEA FY2025/26)
**Official nameplate hydro grid total:** 3,901.366 MW; **solar:** 154.940 MW. The following row totals reproduce your Step 34 exactly, **not** an unsupported national technology split.

- **Recorded ROR-family 1,853.285 MW** includes 904.600 MW with project links still requiring independent verification.
- **Recorded PROR 829.900 MW; Kulekhani I seasonal storage 60 MW.**
- **Cascade 61 MW:** 46 MW Kulekhani II/III linked to Kulekhani I storage water, 15 MW Devighat linked to peaking Trishuli. **Canal Drop 19.2 MW** is treated within ROR; **0.5 MW Pharping** is treated as non-peaking legacy hydro for this scenario.
- **Unclassified 1,077.481 MW** has no source-certified ROR/PROR split. This first run explicitly **assumes it behaves as ROR** (conservative regarding peaking flexibility); changing this scenario is an essential sensitivity, not an official classification.

**Sources:** [NEA A Year in Review FY2025/26](https://www.nea.org.np/en/detail/annual-report-20252026), Appendix 5 p.175 / PDF p.185, Appendix 6 p.177 / PDF p.187 and individual station facts pp.15–24; your Step 34 national ledger and Step 33 technology audit.

In [ ]:
# STEP A: Every existing MW must be accounted for once and only once.
known_ror_family_mw = 1853.285
known_pror_mw = 829.900
kulekhani_I_mw = 60.000
kulekhani_II_III_mw = 46.000
devighat_mw = 15.000
canal_drop_mw = 19.200
pharping_mw = 0.500
unknown_hydro_mw = 1077.481
national_solar_mw = 154.940

# Reconcile official hydropower total BEFORE assigning operating assumptions.
reconciled_hydro = (known_ror_family_mw + known_pror_mw + kulekhani_I_mw
                    + kulekhani_II_III_mw + devighat_mw + canal_drop_mw
                    + pharping_mw + unknown_hydro_mw)
assert abs(reconciled_hydro - 3901.366) < 0.001

# Model allocation: recorded cascade relationships and ONE VISIBLE unknown-type assumption.
UNKNOWN_ASSIGNED_TO_ROR_SHARE = 1.0  # 100% is an explicit SCENARIO, not a verified classification.
existing_ror_mw = (known_ror_family_mw + canal_drop_mw + pharping_mw
                   + UNKNOWN_ASSIGNED_TO_ROR_SHARE * unknown_hydro_mw)
existing_pror_mw = (known_pror_mw + devighat_mw
                    + (1 - UNKNOWN_ASSIGNED_TO_ROR_SHARE) * unknown_hydro_mw)
existing_storage_mw = kulekhani_I_mw + kulekhani_II_III_mw
existing_solar_mw = national_solar_mw

existing = {'ROR': existing_ror_mw, 'PROR': existing_pror_mw,
            'Storage Hydro': existing_storage_mw, 'Solar PV': existing_solar_mw}
assert abs(sum(existing.values()) - (3901.366 + 154.940)) < 0.001

ledger = pd.DataFrame([
    ['Recorded ROR family', known_ror_family_mw, 'Documented/provisional plant classifications'],
    ['Recorded PROR', known_pror_mw, 'NEA-owned / subsidiary classifications'],
    ['Kulekhani I', kulekhani_I_mw, 'NEA source, same reservoir chain'],
    ['Kulekhani II+III', kulekhani_II_III_mw, 'Cascade; water not counted twice'],
    ['Devighat', devighat_mw, 'Trishuli cascade; PROR operational proxy'],
    ['Canal-drop', canal_drop_mw, 'ROR-like availability scenario'],
    ['Pharping', pharping_mw, 'Legacy low-volume hydro; ROR-like proxy'],
    ['Unclassified hydro', unknown_hydro_mw, 'SCENARIO: ROR-like (NOT verified ROR)'],
    ['Solar PV', national_solar_mw, 'NEA national installed capacity'],
], columns=['Source_category','Capacity_MW','Treatment_or_evidence'])
ledger.to_csv(output_folder/'01_existing_capacity_audit.csv', index=False)
print(ledger.to_string(index=False))
print('\n2030 scenario existing capacities (MW):', existing)
print('Official grid-connected hydro reconciled:', round(reconciled_hydro, 3), 'MW')

## 3. Demand — JICA middle-case 2030, source/calculation separated
JICA *Integrated Power System Development Plan* (December 2024) forecasts **24,737 GWh consumer demand**, **13.9% combined system losses**, **4,949 MW generation-side peak** (sources: [Final report, volume 1](https://openjicareport.jica.go.jp/pdf/12387999_01.pdf); [Annexes](https://openjicareport.jica.go.jp/pdf/12387999_02.pdf)).

`Grid demand = consumption / (1 - loss)` = **28,730.546 GWh**. Peak is already sending-end, so do **not** increase 4,949 MW by losses again.

Your Step 37 synthetic two-peak shape is reused, not presented as historic NEA SCADA data. Same daily shape in all 12 months is an explicit simplification. If Step 37 CSV is available on Drive, read and check it; otherwise reconstruct the very same profile.

In [ ]:
# STEP B: Reproduce your Step 36 and Step 37 exactly.
consumer_gwh = 24737.0
loss_fraction = 0.139
peak_mw = 4949.0
annual_grid_gwh = consumer_gwh / (1.0 - loss_fraction)
mean_grid_mw = annual_grid_gwh * 1000 / 8760

shape = np.array([0.53,0.50,0.48,0.47,0.47,0.49,
                  0.59,0.72,0.78,0.76,0.73,0.71,
                  0.71,0.70,0.70,0.73,0.78,0.88,
                  0.95,1.00,0.85,0.75,0.64,0.57])
one_day_mw = (mean_grid_mw + (peak_mw - mean_grid_mw)
              * (shape - shape.mean()) / (shape.max() - shape.mean()))
timestamps = pd.date_range('2030-01-01', periods=8760, freq='h')
demand_8760 = pd.DataFrame({'DateTime': timestamps,
                            'Month': timestamps.month,
                            'Hour': timestamps.hour,
                            'Demand_MW': np.tile(one_day_mw, 365)})

# Use the ACTUAL previously generated file when present, but reject inconsistent data.
drive_demand_path = clean_folder / '2030_Hourly_Demand_v1.csv'
if drive_demand_path.exists():
    supplied = pd.read_csv(drive_demand_path)
    if len(supplied) == 8760 and 'Demand_MW' in supplied.columns:
        if np.allclose(supplied['Demand_MW'].values, demand_8760['Demand_MW'].values, atol=0.01):
            demand_8760['Demand_MW'] = supplied['Demand_MW'].values
            print('Loaded and verified your Step 37 demand CSV.')
        else:
            print('Step 37 file differs from source targets/shape; using auditable reconstruction.')

assert len(demand_8760) == 8760
assert abs(demand_8760['Demand_MW'].sum()/1000 - annual_grid_gwh) < 0.01
assert abs(demand_8760['Demand_MW'].max() - peak_mw) < 0.01

month_days = [31,28,31,30,31,30,31,31,30,31,30,31]
monthly_demand = demand_8760.groupby('Month',as_index=False).agg(
    Hours=('Demand_MW','size'), Energy_MWh=('Demand_MW','sum'), Peak_MW=('Demand_MW','max'))
monthly_demand.to_csv(output_folder/'02_monthly_demand_2030.csv',index=False)
demand_8760.to_csv(output_folder/'02_hourly_demand_2030.csv',index=False)
print('Total 2030 grid-demand energy:',round(annual_grid_gwh,3),'GWh')
print('Peak:',peak_mw,'MW | Minimum:',round(one_day_mw.min(),2),'MW')

## 4. Wet–dry hydrology: use your Step 39–40 **proxy** with dimensionally correct limits
JICA [Final report, volume 1](https://openjicareport.jica.go.jp/pdf/12387999_01.pdf), Section 4.4, gives a planned 2040 portfolio **ROR 14,869 MW / 83,633 GWh** and **PROR 12,831 MW / 63,494 GWh**. These imply annual CFs of **64.21% and 56.49%**—**2040 planning proxies**, not measured current nationwide CFs.

- Our Step 39 monthly ratios are **assumed/interpolated**, anchored to reported wet/dry pattern; they are **not measured plant inflows**.
- ROR: hourly production ≤ available river power = nameplate × monthly availability factor.
- PROR: hourly generation ≤ nameplate, while water inflow enters a **cyclic 24-hour pondage balance** (six full-output-hour equivalent pondage as an **assumption**). Explicit spill prevents creating energy.
- The 12-month representation uses month lengths as weights. Exact calendar February 2030 has 28 days.

In [ ]:
# STEP C: Calibrate seasonal ratios to the JICA portfolio annual CFs.
ratio_ror = np.array([0.4,0.3,0.4,0.6,0.8,1,1,1,1,5/6,2/3,0.5])
ratio_pror = np.array([0.4,0.304,0.4,0.6,0.8,1,1,1,1,5/6,2/3,0.5])
annual_cf_ror = 83633*1000/(14869*8760)
annual_cf_pror = 63494*1000/(12831*8760)

annual_mean_ror = np.average(ratio_ror,weights=month_days)
annual_mean_pror = np.average(ratio_pror,weights=month_days)
ror_cf = annual_cf_ror * ratio_ror / annual_mean_ror
pror_inflow = annual_cf_pror * ratio_pror / annual_mean_pror

assert max(ror_cf) <= 1
assert max(pror_inflow) <= 1
assert abs(np.average(ror_cf,weights=month_days)-annual_cf_ror) < 1e-10
assert abs(np.average(pror_inflow,weights=month_days)-annual_cf_pror) < 1e-10

# User-created Step 40 file is accepted if numerically identical.
drive_hydro_path = clean_folder / '2030_Hydro_Operating_Inputs_v1.csv'
if drive_hydro_path.exists():
    user_hydro = pd.read_csv(drive_hydro_path)
    if len(user_hydro)==8760 and 'ROR_Available_MW_per_MW' in user_hydro.columns:
        if np.allclose(user_hydro['ROR_Available_MW_per_MW'].values,
                       np.repeat(ror_cf, np.array(month_days)*24),atol=1e-8):
            print('Step 40 hydro factors match our source-based reconstruction.')
        else:
            print('WARNING: saved hydro factors differ from expected ratios. Review before publication.')

hydro_monthly = pd.DataFrame({
    'Month':range(1,13), 'Days':month_days,
    'ROR_Season_Ratio':ratio_ror,
    'PROR_Season_Ratio':ratio_pror,
    'ROR_Hourly_Availability_per_MW':ror_cf,
    'PROR_Hourly_Water_Inflow_MWh_per_MW':pror_inflow})
hydro_monthly.to_csv(output_folder/'03_hydropower_calibration.csv',index=False)
print(hydro_monthly.round(3).to_string(index=False))

## 5. Existing storage — use actual NEA station generation, not a made-up reservoir CF
The NEA FY2025/26 Generation Directorate station table (Appendix 6, **PDF page 186 / printed page 176**) reports **92,362.28 MWh** as the year's `Total (STORAGE)` for Kulekhani I–III combined. The monthly totals below come from that table; they **represent observed generation, not reservoir inflows**. We use them as **monthly energy allowances for the existing 106 MW Kulekhani chain**, with no second counting of water at II and III.

NEA reports monthly values in Nepali Shrawan–Ashadh. Mapping each interval to a single English calendar month is an **approximation** (e.g. Poush → January). This should later be upgraded using timestamped plant output. The 0.5 MW Pharping heritage reservoir is included with non-peaking legacy hydro; it is not credited with Kulekhani reservoir energy.

**Future storage** is a *separate candidate* with a 38.43% annual energy yield calculated from JICA’s 2040 portfolio (12,370 MW / 41,638 GWh) and an explicit, limited seasonal reservoir energy inventory. It is not claimed to be a specific commissioned project.

In [ ]:
# STEP D: Real 2025/26 Kulekhani-chain station MWh, remapped approximately to English months.
existing_storage_month_mwh = np.array([
    1943.43, 1446.25, 2914.85, 5124.24, 31370.40, 19079.15,
    10024.79, 13773.60, 4177.10, 726.80, 860.90, 920.77])
assert abs(existing_storage_month_mwh.sum()-92362.28)<0.01
existing_storage_mw = 106.0  # Kulekhani I + II + III: same cascade, NO double water budget

# Candidate new storage: supplied planning CF proxy (not observed 2030 hydro).
new_storage_annual_cf = 41638*1000/(12370*8760)
# Monthly inflow potential has wet-season seasonality. Normalize it by month lengths.
new_storage_month_inflow_per_MW = (
    new_storage_annual_cf * 8760 * ratio_ror * np.array(month_days)
    / sum(ratio_ror*np.array(month_days)))
assert abs(new_storage_month_inflow_per_MW.sum()-new_storage_annual_cf*8760)<1e-7

storage_table = pd.DataFrame({
    'Month': range(1,13), 'NEA_2025_26_Kulekhani_Generation_MWh': existing_storage_month_mwh,
    'New_Storage_Inflow_MWh_per_MW_assumed': new_storage_month_inflow_per_MW})
storage_table.to_csv(output_folder/'04_storage_existing_vs_candidate.csv',index=False)
print('Historical Kulekhani annual generation:',round(existing_storage_month_mwh.sum()/1000,3),'GWh')
print('New-storage annual inflow proxy per MW:',round(new_storage_month_inflow_per_MW.sum(),2),'MWh/MW')

## 6. Solar hourly generation: Nepal planning CF + transparent synthetic diurnal shape
Our [JICA source document](https://openjicareport.jica.go.jp/pdf/12387999_03.pdf) discusses Nepal solar sites and capacity factors in the approximate 15.8–17.5% range. We use **17.0% annual CF** as an editable planning proxy; no satellite hourly irradiance is asserted here. Daylight uses a non-negative sine shape with a noon maximum and zero at night. Wet-season cloud effects are represented by **explicit synthetic monthly weights**. The code normalizes the profile to exactly 17% annual energy per installed MW.

**Next research upgrade:** replace this profile with NASA POWER / Global Solar Atlas or PVGIS *site-specific time series*, then verify national weighting by potential deployment region. A Kathmandu-only weather series is not a national deployment-weighted solar resource.

In [ ]:
# STEP E: A 24-hour PV shape and seasonal cloudiness scenario.
solar_cf_annual = 0.17  # JICA Nepal planning-range proxy; not observed PV hourly output.
solar_month_weights = np.array([0.90,1.02,1.12,1.13,1.05,0.88,0.72,0.75,0.86,1.00,1.06,1.00])
solar_diurnal = np.array([max(0.0,np.sin(np.pi*(h-6)/12)) for h in range(24)])
# Calibrate such that sum over all hours / 8760 = 0.17.
weighted_mean = (sum(month_days[m]*solar_month_weights[m] for m in range(12))
                 /365) * solar_diurnal.mean()
solar_scale = solar_cf_annual / weighted_mean
solar_profile = np.array([solar_scale*w*solar_diurnal for w in solar_month_weights])
assert solar_profile.shape == (12,24)
assert solar_profile.min() >= 0 and solar_profile.max() <= 1
assert abs(sum(month_days[m]*solar_profile[m].sum() for m in range(12))/8760 - solar_cf_annual) < 1e-10

solar_table = pd.DataFrame(solar_profile,
    index=pd.Index(range(1,13),name='Month'), columns=[f'Hour_{h:02d}' for h in range(24)])
solar_table.to_csv(output_folder/'05_solar_24h_by_month_assumed.csv')
print('Annual solar capacity factor:',round(solar_cf_annual*100,2),'%')
print('Maximum hourly solar availability:',round(solar_profile.max(),3),'MW per MW installed')
print('Solar zero at 00:00:',np.allclose(solar_profile[:,0],0))

## 7. Technology costs, policy and source-quality ledger
The six-row `nepal_jica_cost_lcoe_parameters.csv` supplied in your project gives technology benchmarks. **We use its ROR/PROR/storage/solar cost rows as planning assumptions**, not certified NEA/JICA procurement prices. FOM, variable O&M, plant life and CAPEX below are copied from that table **when available**, otherwise the same values are included as an embedded fallback for a single-file notebook.

**Important:** This model uses **variable operating cost per MWh in the objective** (analogous mathematically to a thermal generator's per-unit fuel cost), but *hydro and solar are not fossil-fuel plants*. **PPA purchase tariffs must not also be counted as generation operating costs** on top of annualized new CAPEX—that would double-count investment recovery.

Currency is **NPR million/year**. The **NPR/USD exchange rate 145** is a *modeling assumption*, not a claimed NRB spot rate. The discount rate (8%) is an assumption. **Reliability VoLL** uses a historical 0.76 USD/kWh energy-not-supplied cost quoted in JICA Annex 4 (from an earlier Master Plan, NOT a current 2030 value). **Imports** use the FY2025/26 NEA report's approximately NPR 10.56 billion import cost / 1,171 GWh imported = **9.02 NPR/kWh historical average**, not a guaranteed 2030 market price. The 750 MW import ceiling is a *conservative 2030 scenario*, informed by NEA's FY2025/26 approximate import capacity, not an announced 2030 firm treaty.

**Expansion ceilings** from your `nepal_jica_expansion_scenarios.csv` are **indicative 2030 portfolio screens**, not guaranteed site-ready buildable limits or confirmed commissioning dates. We subtract 2026 existing MW, so the upper bound is applied to *new* construction only.

In [ ]:
# STEP F: Technology costs. Defaults are exactly the entries in your provided cost CSV.
technologies = ['ROR','PROR','Storage Hydro','Solar PV']
tech_cost = pd.DataFrame([
    ['ROR',1650,35,1.5,40],
    ['PROR',1850,40,2.0,45],
    ['Storage Hydro',2500,50,2.5,50],
    ['Solar PV',691,12,0.0,25],
],columns=['Technology','CAPEX_USD_per_kW','Fixed_OM_USD_per_kW_year',
           'Variable_OM_USD_per_MWh','Lifetime_years']).set_index('Technology')

cost_file = input_folder/'nepal_jica_cost_lcoe_parameters.csv'
if cost_file.exists():
    provided = pd.read_csv(cost_file)
    csv_map = {'ROR':'Run-of-River (ROR) Hydro',
               'PROR':'Peaking ROR (PROR) Hydro',
               'Storage Hydro':'Storage / Reservoir Hydro',
               'Solar PV':'Utility Solar PV'}
    for tech,label in csv_map.items():
        rows = provided[provided['Technology'].str.strip()==label]
        if len(rows)==1:
            row=rows.iloc[0]
            tech_cost.loc[tech,'CAPEX_USD_per_kW']=row['Capital_Cost_USD_per_kW']
            tech_cost.loc[tech,'Fixed_OM_USD_per_kW_year']=row['Fixed_OM_USD_per_kW_yr']
            tech_cost.loc[tech,'Variable_OM_USD_per_MWh']=row['Var_OM_USD_per_MWh']
            tech_cost.loc[tech,'Lifetime_years']=row['Economic_Life_Years']
    print('Loaded technology benchmarks from your provided project CSV.')
else:
    print('Used the identical embedded cost-table benchmarks; CSV not mounted.')

NPR_PER_USD = 145.0  # EXPLICIT USER-EDITABLE SCENARIO, not a verified exchange rate
DISCOUNT_RATE = 0.08 # EXPLICIT USER-EDITABLE SCENARIO
IMPORT_PRICE_NPR_per_kWh = 10560.0/1171.0  # million NPR / GWh = NPR/kWh
IMPORT_LIMIT_MW = 750.0  # 2030 scenario; NEA observed ~750MW imports FY25/26
VOLL_NPR_per_kWh = 0.76 * NPR_PER_USD  # JICA Annex 4 quotes a HISTORICAL MP2014 ENS cost; 2030 proxy
MAX_ENS_FRACTION = 0.0001 # 0.01% deterministic energy-not-served limit; ASSUMPTION
FIRM_CAPACITY_MARGIN = 0.05 # 5% dry-peak planning test; NOT a probabilistic LOLE standard
PROR_PONDAGE_HOURS = 6.0 # illustrative, NOT an audited national pondage volume
NEW_STORAGE_ENERGY_HOURS = 900.0 # full-output equivalent reservoir MWh/MW: ASSUMPTION

# Upper capacity screen based on the supplied 2030 JICA-like scenario column, not guaranteed sites.
scenario_file = input_folder/'nepal_jica_expansion_scenarios.csv'
capacity_screen = {'ROR':7843.7,'PROR':3711.6,'Storage Hydro':781.4,'Solar PV':2182.7}
if scenario_file.exists():
    scen=pd.read_csv(scenario_file)
    record=scen.loc[scen['Year']==2030]
    assert len(record)==1
    record=record.iloc[0]
    capacity_screen={'ROR':float(record['ROR_MW']),
                     'PROR':float(record['PROR_MW']),
                     'Storage Hydro':float(record['Storage_MW']),
                     'Solar PV':float(record['Solar_MW'])}

max_new = {name:max(0.0, capacity_screen[name]-existing[name]) for name in technologies}
for name in technologies:
    assert max_new[name]>0, 'No buildable capacity screen for '+name

# Calculate CRF and annualized cost in million NPR per MW.
annual_capex_million_per_MW={}
fixed_om_million_per_MW={}
variable_npr_per_MWh={}
for tech in technologies:
    r=DISCOUNT_RATE
    n=int(tech_cost.loc[tech,'Lifetime_years'])
    crf=r*(1+r)**n/((1+r)**n-1)
    annual_capex_million_per_MW[tech] = (tech_cost.loc[tech,'CAPEX_USD_per_kW']
                                        *1000*NPR_PER_USD*crf/1e6)
    fixed_om_million_per_MW[tech] = (tech_cost.loc[tech,'Fixed_OM_USD_per_kW_year']
                                    *1000*NPR_PER_USD/1e6)
    variable_npr_per_MWh[tech] = tech_cost.loc[tech,'Variable_OM_USD_per_MWh']*NPR_PER_USD

cost_export=tech_cost.reset_index().copy()
cost_export['Evidence_Status']='Supplied planning CSV benchmark, not independently verified for all rows'
cost_export.to_csv(output_folder/'06_technology_cost_assumptions.csv',index=False)
print('New buildable screening caps (MW):', {k:round(v,2) for k,v in max_new.items()})
print('Historical NEA implied import price:',round(IMPORT_PRICE_NPR_per_kWh,3),'NPR/kWh')
print('Annualized new-build costs (million NPR per MW):',
      {k:round(v,2) for k,v in annual_capex_million_per_MW.items()})

## 8. Mathematical optimization formulation — least-cost linear program

Each month has one representative 24-hour day, repeated for the number of calendar days. Index **m=1…12**, **h=0…23**, and let \(d_m\) be days in month m.

**Decisions:** \(x_g\) new MW for ROR, PROR, storage and solar; \(p_{g,m,h}\) MW dispatched; \(i_{m,h}\) imported MW; \(u_{m,h}\) unserved MW; \(w_{m,h}\) PROR spill; \(S_{m,h}\) pondage MWh; \(R_m\) new reservoir MWh and \(z_m\) reservoir spill MWh.

**Objective (NPR million per year):**

\[
\min Z=\underbrace{\sum_g (CAPEX_g\,CRF_g+FOM_g)x_g}_{\text{annualized new build + fixed O&M}}
+\underbrace{\sum_{m,h}d_m\left[\sum_g VOM_g p_{g,m,h}+c_I i_{m,h}+VOLL\,u_{m,h}\right]}_{\text{annual variable costs and unserved-energy cost}}
+\text{fixed O&M of existing capacity}.
\]

All terms are expressed consistently in **million NPR**, not USD mixed with NPR. Fixed O&M on existing plants is a constant included in the *reported* objective value.

**Main constraints:**

1. \(p_{ROR}+p_{PROR}+p_{STOexisting}+p_{STOnew}+p_{PV}+i+u=D_{m,h}\) (hourly supply/demand).
2. \(p_{ROR,m,h}\le (K_{ROR}^{2026}+x_{ROR})a^{ROR}_m\).
3. \(p_{PV,m,h}\le (K_{PV}^{2026}+x_{PV})a^{PV}_{m,h}\).
4. \(p_{PROR,m,h}\le K_{PROR}^{2026}+x_{PROR}\), with **hourly water balance** \(S_{h+1}=S_h+inflow_m(K+x)-p_{PROR,h}-spill_h\), \(0\le S_h\le6(K+x)\), and \(S_{0}=S_{24}\) for each representative day.
5. Existing Kulekhani cascade \(p_{STOexisting}\le106\) MW, and \(d_m\sum_h p_{STOexisting,m,h}\le E^{NEA2026}_{m}\) (actual historical 2026 energy as proxy).
6. New reservoir generation \(p_{STOnew}\le x_{STO}\), with **monthly reservoir balance** \(R_{m+1}=R_m+inflow_m x_{STO}-d_m\sum_h p_{STOnew,m,h}-z_m\), \(0\le R_m\le900 x_{STO}\), and cyclic annual boundary.
7. \(0\le i_{m,h}\le750\) MW; \(\sum d_m u_{m,h}\le0.01\%\, E_{annual}\); dry-peak capacity proxy with 5% margin; and \(0\le x_g\le x_g^{max}\).

Solar and ROR curtailment are the difference between **available** and **dispatched** power; we report them separately. **ENS is deterministic**, not probabilistic *expected* ENS without outage probabilities. **VOLL is a penalty to value reliability**, not an imported tariff. These modeling choices and limits are scenario assumptions.

## 9. Small solver helper — builds a sparse linear program
You do not need to edit this cell. The optimization equations are added in the next two cells as ordinary dictionaries: `{variable: coefficient}`. These map exactly to the equations written above. SciPy's built-in HiGHS solver makes this notebook independent of PuLP version problems.

In [ ]:
# STEP G: A compact way to specify an LP using SciPy.
# Variable numbers point to their position in the solver vector.
var_names=[]
var_cost=[]
var_bounds=[]
rows_eq=[]
rhs_eq=[]
rows_le=[]
rhs_le=[]

def variable(name, cost=0.0, upper=None):
    number=len(var_names)
    var_names.append(name)
    var_cost.append(float(cost))
    var_bounds.append((0.0, upper))
    return number

def equal(terms, target):
    rows_eq.append(terms)
    rhs_eq.append(float(target))

def at_most(terms, target):
    rows_le.append(terms)
    rhs_le.append(float(target))

# Four investment variables (new MW). Annualized cost + fixed O&M are both paid on NEW MW.
new={}
for tech in technologies:
    yearly_million=(annual_capex_million_per_MW[tech]
                    +fixed_om_million_per_MW[tech])
    new[tech]=variable('New_'+tech, yearly_million, max_new[tech])

print('New-capacity decision variables:',len(new))

## 10. Add hourly dispatch, short-term pondage and long-term reservoir variables
Every variable's annual operating cost is weighted by the number of days in its month. The reservoir state variables have zero direct objective coefficient; they exist to enforce conservation of stored water.

In [ ]:
# STEP H: Create hourly decision variables for all 12 representative days.
ror={}; pror={}; storage_old={}; storage_new={}; solar={}; imports={}; ens={}
pror_spill={}; pond={}; reservoir={}; reservoir_spill={}
for m in range(12):
    d=month_days[m]
    for h in range(24):
        ror[m,h]=variable(f'ROR_{m+1}_{h}', d*variable_npr_per_MWh['ROR']/1e6)
        pror[m,h]=variable(f'PROR_{m+1}_{h}', d*variable_npr_per_MWh['PROR']/1e6)
        storage_old[m,h]=variable(f'ExistingStorage_{m+1}_{h}',
                                  d*variable_npr_per_MWh['Storage Hydro']/1e6)
        storage_new[m,h]=variable(f'NewStorage_{m+1}_{h}',
                                  d*variable_npr_per_MWh['Storage Hydro']/1e6)
        solar[m,h]=variable(f'Solar_{m+1}_{h}',d*variable_npr_per_MWh['Solar PV']/1e6)
        imports[m,h]=variable(f'Import_{m+1}_{h}',
                              d*IMPORT_PRICE_NPR_per_kWh*1000/1e6, IMPORT_LIMIT_MW)
        ens[m,h]=variable(f'ENS_{m+1}_{h}', d*VOLL_NPR_per_kWh*1000/1e6)
        pror_spill[m,h]=variable(f'PROR_spill_{m+1}_{h}')
    for h in range(25):
        pond[m,h]=variable(f'Pondage_MWh_{m+1}_{h}')
for m in range(13):
    reservoir[m]=variable(f'SeasonalReservoir_MWh_{m}')
for m in range(12):
    reservoir_spill[m]=variable(f'Reservoir_spill_MWh_{m+1}')

print('Total LP variables:',len(var_names))

## 11. Add hourly constraints in readable form
The function names `equal()` and `at_most()` mean exactly = and ≤. Every line is a linear equation: terms with negative coefficients represent a decision on the right side after rearrangement.

In [ ]:
# STEP I: Hourly supply balance, generation bounds and PROR pondage balance.
for m in range(12):
    for h in range(24):
        # 1. All delivered sources plus unserved energy meet the hour's demand.
        equal({ror[m,h]:1, pror[m,h]:1, storage_old[m,h]:1,
               storage_new[m,h]:1, solar[m,h]:1,
               imports[m,h]:1, ens[m,h]:1}, float(one_day_mw[h]))

        # 2. ROR output <= monthly available river output times total installed ROR.
        at_most({ror[m,h]:1, new['ROR']:-float(ror_cf[m])},
                float(ror_cf[m]*existing['ROR']))

        # 3. Solar can never produce more than sunshine permits.
        at_most({solar[m,h]:1, new['Solar PV']:-float(solar_profile[m,h])},
                float(solar_profile[m,h]*existing['Solar PV']))

        # 4. PROR turbine output cannot exceed nameplate power.
        at_most({pror[m,h]:1, new['PROR']:-1}, existing['PROR'])

        # 5. Existing Kulekhani I+II+III summed turbine capacity, but one water budget.
        at_most({storage_old[m,h]:1}, existing_storage_mw)

        # 6. New seasonal-storage turbine cannot exceed capacity selected by optimization.
        at_most({storage_new[m,h]:1, new['Storage Hydro']:-1},0)

        # 7. PROR water conservation (1 hour): end = start + inflow - generation - spillage.
        inflow=float(pror_inflow[m])
        equal({pond[m,h+1]:1, pond[m,h]:-1,
               pror[m,h]:1, pror_spill[m,h]:1,
               new['PROR']:-inflow}, inflow*existing['PROR'])

    # 8. Capacity of pondage = 6-hour equivalent full-power output (scenario parameter).
    for h in range(25):
        at_most({pond[m,h]:1, new['PROR']:-PROR_PONDAGE_HOURS},
                PROR_PONDAGE_HOURS*existing['PROR'])

    # Start at 50% pondage and return to the same level at end of representative day.
    equal({pond[m,0]:1,new['PROR']:-PROR_PONDAGE_HOURS/2},
          PROR_PONDAGE_HOURS/2*existing['PROR'])
    equal({pond[m,24]:1,pond[m,0]:-1},0)

    # 9. Historical existing reservoir-chain monthly electricity budget.
    at_most({storage_old[m,h]:month_days[m] for h in range(24)},
            float(existing_storage_month_mwh[m]))

print('Hourly/pondage constraints entered.')

## 12. Annual seasonal storage, reliability and reserve constraints
The candidate reservoir has a **month-to-month energy balance**, rather than unlimited daily generation. For the initial demonstration its usable inventory is **900 MWh per MW of new storage capacity** (an explicit, technology/site-dependent assumption requiring candidate feasibility data). We restrict accumulated reservoir energy and ensure the end-of-year inventory returns to the initial inventory.

Existing Kulekhani's historical monthly energy budget is kept separate, so it is **not** credited with this assumed 900-hour new-build reservoir.

In [ ]:
# STEP J: New-storage reservoir conservation and reliability.
for m in range(12):
    terms={reservoir[m+1]:1, reservoir[m]:-1, reservoir_spill[m]:1,
           new['Storage Hydro']:-float(new_storage_month_inflow_per_MW[m])}
    for h in range(24):
        terms[storage_new[m,h]]=month_days[m]
    equal(terms,0)

for m in range(13):
    at_most({reservoir[m]:1,new['Storage Hydro']:-NEW_STORAGE_ENERGY_HOURS},0)

# Begin at half reservoir and finish year at same state (no free stored water).
equal({reservoir[0]:1,new['Storage Hydro']:-NEW_STORAGE_ENERGY_HOURS/2},0)
equal({reservoir[12]:1,reservoir[0]:-1},0)

# Reliability: deterministic annual energy-not-served budget, not probabilistic EENS.
ens_budget_mwh = MAX_ENS_FRACTION*annual_grid_gwh*1000
at_most({ens[m,h]:month_days[m] for m in range(12) for h in range(24)},
        ens_budget_mwh)

# Approximate 5% dry evening peak capacity test (explicit scenario proxy).
# February is our representative lowest-water month.
firm_ror=float(ror_cf[1])
required_firm=(1+FIRM_CAPACITY_MARGIN)*peak_mw
constant_firm=(firm_ror*existing['ROR']+existing['PROR']
               +existing_storage_mw+IMPORT_LIMIT_MW)
# Minus total installed firm capacity <= -required firm capacity.
at_most({new['ROR']:-firm_ror,new['PROR']:-1,new['Storage Hydro']:-1},
        constant_firm-required_firm)

print('Daily pondage, monthly reservoir, ENS and dry-peak adequacy constraints complete.')
print('Constraint counts:',len(rows_eq),'equalities,',len(rows_le),'upper bounds.')

## 13. Solve with SciPy HiGHS (included in Colab)
We convert the sparse coefficient dictionaries into solver matrices, call `linprog`, and **stop on a non-optimal status**. We add the existing fleet's annual fixed O&M afterward, because it is constant with respect to investment decisions but belongs in the system cost accounting.

In [ ]:
# STEP K: Convert our explicit equations into sparse matrices.
n_variables=len(var_names)

def make_matrix(rows):
    ii=[]; jj=[]; vv=[]
    for row_no, terms in enumerate(rows):
        for column_no, coefficient in terms.items():
            if coefficient!=0:
                ii.append(row_no)
                jj.append(column_no)
                vv.append(float(coefficient))
    return coo_matrix((vv,(ii,jj)),shape=(len(rows),n_variables)).tocsr()

A_eq=make_matrix(rows_eq)
A_ub=make_matrix(rows_le)
result=linprog(c=np.array(var_cost),A_ub=A_ub,b_ub=np.array(rhs_le),
               A_eq=A_eq,b_eq=np.array(rhs_eq),bounds=var_bounds,
               method='highs',options={'primal_feasibility_tolerance':1e-7,
                                       'dual_feasibility_tolerance':1e-7})

print('Solver message:',result.message)
assert result.success, 'Model did not solve optimally. Review limits and constraints.'
y=result.x

# Cost of historical existing fixed O&M is a constant, not paid again as new CAPEX.
fixed_om_existing = sum(existing[t]*fixed_om_million_per_MW[t] for t in technologies)
annual_system_cost_million = float(result.fun + fixed_om_existing)
print('OPTIMAL — annual system cost:',round(annual_system_cost_million,2),'million NPR/year')
print('Core variable cost + new investment:',round(result.fun,2),'million NPR/year')
print('Historical fleet fixed O&M:',round(fixed_om_existing,2),'million NPR/year')

## 14. Optimal 2030 capacity mix, technology-by-technology
**Existing MW** is the modeled 2026 classification scenario, not an unqualified official technology split. **New MW** is determined by the solver, bounded by the illustrative 2030 planning capacity ceilings. This is the headline *capacity* result.

In [ ]:
# STEP L: Summarize capacities and show maximum limits.
capacity_rows=[]
for tech in technologies:
    added=float(y[new[tech]])
    base=float(existing[tech])
    capacity_rows.append({
        'Technology':tech, 'Existing_2026_MW_scenario':base,
        'Optimized_New_MW':added, 'Total_2030_MW':base+added,
        'Max_New_MW_assumption':max_new[tech]})
capacity_mix=pd.DataFrame(capacity_rows)
capacity_mix['2030_Capacity_Share_pct']=100*capacity_mix['Total_2030_MW']/capacity_mix['Total_2030_MW'].sum()
print(capacity_mix.round(3).to_string(index=False))
print('\n2030 total installed capacity:',round(capacity_mix['Total_2030_MW'].sum(),2),'MW')
for _, rec in capacity_mix.iterrows():
    if abs(rec['Optimized_New_MW']-rec['Max_New_MW_assumption'])<0.01:
        print('CAUTION: New capacity ceiling is binding for',rec['Technology'],
              '- actual 2030 commissioning feasibility is not established.')
capacity_mix.to_csv(output_folder/'07_optimal_capacity_mix_2030.csv',index=False)

## 15. Dispatch for every representative hour, annual energy and energy mix
MW multiplied by one hour gives MWh; multiplying a representative day's energy by days in the month gives monthly MWh. **Electricity consumed by domestic demand includes imports**, so the report gives both (a) *domestic generation mix* and (b) *total supply mix including imports*.

Hydro/solar potential not dispatched becomes **curtailed energy** (calculated explicitly rather than silently omitted).

In [ ]:
# STEP M: Read solver outputs into one transparent 288-row table.
dispatch_rows=[]
for m in range(12):
    for h in range(24):
        r=float(y[ror[m,h]])
        p=float(y[pror[m,h]])
        e=float(y[storage_old[m,h]])
        n=float(y[storage_new[m,h]])
        s=float(y[solar[m,h]])
        imp=float(y[imports[m,h]])
        unmet=float(y[ens[m,h]])
        ror_av=float(ror_cf[m]*(existing['ROR']+y[new['ROR']]))
        pv_av=float(solar_profile[m,h]*(existing['Solar PV']+y[new['Solar PV']]))
        dispatch_rows.append({
            'Month':m+1,'Hour':h,'Days_in_Month':month_days[m],
            'Demand_MW':float(one_day_mw[h]), 'ROR_MW':r,'PROR_MW':p,
            'Existing_Storage_MW':e,'New_Storage_MW':n,'Storage_Hydro_MW':e+n,
            'Solar_PV_MW':s,'Imports_MW':imp,'Unserved_MW':unmet,
            'ROR_Curtailed_MW':max(0,ror_av-r),
            'Solar_Curtailed_MW':max(0,pv_av-s),
            'PROR_Spilled_Water_Equivalent_MWh_per_hour':float(y[pror_spill[m,h]])})
dispatch_288=pd.DataFrame(dispatch_rows)
dispatch_288.to_csv(output_folder/'08_dispatch_288_representative_hours.csv',index=False)

energy_names={'ROR':'ROR_MW','PROR':'PROR_MW','Storage Hydro':'Storage_Hydro_MW',
              'Solar PV':'Solar_PV_MW','Imports':'Imports_MW','Unserved':'Unserved_MW'}
energy_rows=[]
for name,column in energy_names.items():
    mwh=float((dispatch_288[column]*dispatch_288['Days_in_Month']).sum())
    energy_rows.append({'Resource':name,'Energy_GWh':mwh/1000})
energy_mix=pd.DataFrame(energy_rows)
energy_mix['Share_of_Annual_Demand_pct']=100*energy_mix['Energy_GWh']/annual_grid_gwh

domestic_generation_gwh=energy_mix.loc[energy_mix['Resource'].isin(technologies),'Energy_GWh'].sum()
energy_mix['Share_of_Domestic_Generation_pct']=np.where(
    energy_mix['Resource'].isin(technologies),
    100*energy_mix['Energy_GWh']/domestic_generation_gwh, 0)

assert abs(energy_mix['Energy_GWh'].sum()-annual_grid_gwh)<0.01
print('ANNUAL 2030 ELECTRICITY SUPPLY MIX')
print(energy_mix.round(3).to_string(index=False))
print('\nDomestic generation:',round(domestic_generation_gwh,3),'GWh')
print('Hydro + PV curtailment:',round(sum((dispatch_288[c]*dispatch_288.Days_in_Month).sum()/1000
                                   for c in ['ROR_Curtailed_MW','Solar_Curtailed_MW']),3),'GWh')
energy_mix.to_csv(output_folder/'09_optimal_energy_mix_2030.csv',index=False)

## 16. Two requested dispatch plots — dry vs wet season
February = lowest modeled ROR availability; August = wet-season maximum. The stacked supply should exactly meet the demand line, including any (highly penalized) imports or unserved energy. No fabricated plots are included; these are drawn from solver results.

In [ ]:
# STEP N: Draw dry and wet season one-day dispatch directly from the solver result.
plot_resources = [('ROR_MW','ROR'),('PROR_MW','PROR'),
                  ('Storage_Hydro_MW','Storage Hydro'),('Solar_PV_MW','Solar PV'),
                  ('Imports_MW','Imports'),('Unserved_MW','Unserved demand')]
for month,label in [(2,'Dry season — February'),(8,'Wet season — August')]:
    day=dispatch_288[dispatch_288['Month']==month].sort_values('Hour')
    x=day['Hour'].values
    series=[day[col].values for col,_ in plot_resources]
    plt.figure(figsize=(11,5))
    plt.stackplot(x,*series,labels=[name for _,name in plot_resources],alpha=0.85)
    plt.plot(x,day['Demand_MW'].values,color='black',linewidth=2.0,label='Demand')
    plt.title(f'Nepal GEP 2030 — {label} (modeled dispatch)')
    plt.xlabel('Hour of day (Nepal local time)')
    plt.ylabel('MW')
    plt.xticks(range(0,24,2))
    plt.grid(alpha=0.2)
    plt.legend(ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.14),frameon=False)
    plt.tight_layout()
    fig_name=('10_dispatch_dry_day.png' if month==2 else '11_dispatch_wet_day.png')
    plt.savefig(output_folder/fig_name,dpi=180,bbox_inches='tight')
    plt.show()
print('Two dispatch figures saved.')

## 17. Reconstruct a complete 8,760-hour dispatch CSV
Because the demand, hydro availability and solar shape repeat **exactly** within a month, a 24-hour monthly dispatch multiplied by the month's number of days is mathematically consistent with this simplified annual representation. This is **not 8,760 independent optimization hours**; a future model with real daily weather or reservoir inflows must optimize the actual chronological hours.

In [ ]:
# STEP O: Map solved representative hourly output onto the 2030 calendar.
full_year=demand_8760.merge(dispatch_288.drop(columns=['Demand_MW']),on=['Month','Hour'],how='left',validate='many_to_one')
assert len(full_year)==8760
assert not full_year.isna().any().any()
full_year['Supply_MW']=(full_year[['ROR_MW','PROR_MW','Storage_Hydro_MW',
                                   'Solar_PV_MW','Imports_MW','Unserved_MW']].sum(axis=1))

max_balance_error=(full_year['Supply_MW']-full_year['Demand_MW']).abs().max()
assert max_balance_error < 0.01
full_year.to_csv(output_folder/'12_2030_dispatch_all_8760_hours.csv',index=False)
print('Hourly rows:',len(full_year))
print('Largest hourly supply-demand difference:',max_balance_error,'MW')
print('Full-year reconstructed dispatch saved.')

## 18. Cost accounting — explain why the optimizer selected the mix
Costs below are **annual**, **NPR million/year**, and include: annualized CAPEX of **new build**, fixed O&M on **all** installed capacity, variable dispatch cost of generating MWh, historical-price imports, and reliability/VOLL cost of unserved MWh. There is **no second charge for PPA tariffs**, which would double count.

In [ ]:
# STEP P: Reconstruct objective from transparent cost categories.
new_investment_mn=sum(float(y[new[t]])*annual_capex_million_per_MW[t] for t in technologies)
fixed_om_mn=sum((existing[t]+float(y[new[t]]))*fixed_om_million_per_MW[t] for t in technologies)
variable_om_mn=sum(float(energy_mix.loc[energy_mix.Resource==t,'Energy_GWh'].iloc[0])*1000
                   *variable_npr_per_MWh[t]/1e6 for t in technologies)
import_mn=float(energy_mix.loc[energy_mix.Resource=='Imports','Energy_GWh'].iloc[0])*1000 \
          *IMPORT_PRICE_NPR_per_kWh*1000/1e6
voll_mn=float(energy_mix.loc[energy_mix.Resource=='Unserved','Energy_GWh'].iloc[0])*1000 \
        *VOLL_NPR_per_kWh*1000/1e6
costs=pd.DataFrame([
    ['Annualized new investment',new_investment_mn],
    ['Fixed O&M (all installed)',fixed_om_mn],
    ['Variable generation O&M',variable_om_mn],
    ['Imports at FY2025/26 implied average price',import_mn],
    ['Value of unserved demand (VOLL scenario)',voll_mn],
],columns=['Cost_Component','Million_NPR_per_year'])
assert abs(costs.Million_NPR_per_year.sum()-annual_system_cost_million)<0.05
costs.to_csv(output_folder/'13_annual_system_cost_breakdown.csv',index=False)
print(costs.round(3).to_string(index=False))
print('TOTAL:',round(annual_system_cost_million,2),'million NPR/year')

## 19. Automatic physical, mathematical and source-integrity checks
**Optimal** does not guarantee good input assumptions, so we test every operational constraint independently. We also check that the 2026 hydro baseline is conserved, and report remaining evidence gaps. A deterministically calculated ENS cannot be called probabilistic expected ENS without outage statistics.

In [ ]:
# STEP Q: Cross-check everything important.
checks=[]
def check(label,passed,detail):
    checks.append({'Check':label,'Passed':bool(passed),'Detail':str(detail)})

check('NEA grid-connected hydro = 3901.366 MW',abs(reconciled_hydro-3901.366)<1e-3,reconciled_hydro)
check('2026 national solar = 154.940 MW',abs(existing['Solar PV']-154.94)<1e-3,existing['Solar PV'])
check('Annual grid demand = JICA loss-adjusted target',
      abs(full_year.Demand_MW.sum()/1000 - annual_grid_gwh)<0.01,
      full_year.Demand_MW.sum()/1000)
check('Hourly balance',max_balance_error<0.01,max_balance_error)
check('Import capacity ceiling',full_year.Imports_MW.max()<=IMPORT_LIMIT_MW+1e-4,
      full_year.Imports_MW.max())
check('ROR availability ceiling',
      all(dispatch_288[(dispatch_288.Month==m+1)].ROR_MW.max()<=
          ror_cf[m]*(existing['ROR']+y[new['ROR']])+1e-4 for m in range(12)),
      '12 monthly checks')
check('Solar availability ceiling',
      all(row['Solar_PV_MW'] <= solar_profile[int(row.Month)-1,int(row.Hour)]*
          (existing['Solar PV']+y[new['Solar PV']])+1e-4
          for _,row in dispatch_288.iterrows()),'288 hourly checks')
check('Existing storage never exceeds historical monthly energy',
      all(month_days[m]*dispatch_288[dispatch_288.Month==m+1].Existing_Storage_MW.sum()<=
          existing_storage_month_mwh[m]+1e-3 for m in range(12)),'12 monthly checks')
check('PROR pondage cyclic per day',
      all(abs(y[pond[m,24]]-y[pond[m,0]])<1e-4 for m in range(12)),
      '12 daily pondage boundaries')
check('Seasonal new storage cyclic per year',
      abs(y[reservoir[12]]-y[reservoir[0]])<1e-4,
      'start/end stored MWh')
check('Deterministic ENS <= annual limit',
      energy_mix.loc[energy_mix.Resource=='Unserved','Energy_GWh'].iloc[0]*1000<=ens_budget_mwh+1e-3,
      f'ENS budget {ens_budget_mwh:.1f} MWh')
check('New capacities within scenario ceilings',
      all(y[new[t]]<=max_new[t]+1e-5 for t in technologies),max_new)

validation=pd.DataFrame(checks)
print(validation.to_string(index=False))
assert validation['Passed'].all(), 'One or more validation checks failed.'
validation.to_csv(output_folder/'14_validation_checks.csv',index=False)
print('\nALL VALIDATION CHECKS PASSED')

## 20. Export a source-and-assumption register — critical for the paper
Each important input is tagged as **published source / arithmetic calculation / synthetic assumption / unverified classification**. Keep this file with the model version when publishing results or pushing to your existing Nepal GEP GitHub workspace.

This notebook intentionally does **not** fabricate BESS efficiencies, outage probabilities, detailed new-reservoir specifications, hourly Nepal-wide solar observations or 2030 commissioning dates. Those are separate research inputs for a future higher-fidelity model.

In [ ]:
# STEP R: Metadata and provenance, with transparent limitations.
audit_items = [
    ('Existing hydro total',3901.366,'MW','NEA FY2025/26 Appendix 5','Published source'),
    ('Existing solar total',154.940,'MW','NEA FY2025/26 Appendix 5','Published source'),
    ('Unclassified existing hydro',1077.481,'MW','Your audited Step 34 ledger','Not yet technology-verified'),
    ('Unknown-hydro share treated as ROR',UNKNOWN_ASSIGNED_TO_ROR_SHARE,'fraction',
     'Explicit illustrative dispatch scenario','Assumption / classification uncertainty'),
    ('IPP ROR-family links still awaiting source recheck',904.600,'MW',
     'Step 31 independent source-quality audit','Provisional classification'),
    ('2030 consumer electricity',24737.0,'GWh','JICA IPSDP 2024, middle-case','Published source'),
    ('2030 system loss',13.9,'%','JICA IPSDP Annex 5','Published source'),
    ('2030 peak demand',4949.0,'MW','JICA IPSDP middle-case','Published source'),
    ('2030 hourly daily demand','24 values','hourly shape','User Step 37','Synthetic/assumption'),
    ('ROR 2040 planning annual CF',round(annual_cf_ror,5),'fraction',
     'JICA IPSDP 2040 candidate project portfolio','Calculated proxy for 2030'),
    ('PROR 2040 planning annual CF',round(annual_cf_pror,5),'fraction',
     'JICA IPSDP 2040 candidate project portfolio','Calculated proxy for 2030'),
    ('Existing Kulekhani I-II-III annual output',92362.28,'MWh',
     'NEA FY2025/26 Generation Directorate monthly station table','Observed historical generation'),
    ('Solar annual capacity factor',solar_cf_annual,'fraction',
     'JICA Nepal solar range','Planning proxy'),
    ('Solar 24-hour/12-month coefficients','synthetic','per unit',
     'This notebook; not NASA plant time-series','Synthetic/assumption'),
    ('New storage annual capacity factor',new_storage_annual_cf,'fraction',
     'JICA IPSDP 2040 STO portfolio 41,638 GWh / 12,370 MW','Calculated proxy, not a 2030 reservoir forecast'),
    ('New storage energy duration',NEW_STORAGE_ENERGY_HOURS,'hours',
     'Illustrative candidate seasonal storage','Assumption'),
    ('PROR pondage duration',PROR_PONDAGE_HOURS,'hours',
     'NREL/JICA style short-term peaking model','Assumption, no national measured duration'),
    ('FX rate',NPR_PER_USD,'NPR/USD','No NRB historical conversion fixed','Explicit assumption'),
    ('Discount rate',DISCOUNT_RATE,'fraction','No source-specific discount rate applied','Explicit assumption'),
    ('Import price',IMPORT_PRICE_NPR_per_kWh,'NPR/kWh',
     'NEA FY2025/26 import cost/energy ratio','Derived historical proxy'),
    ('Import limit',IMPORT_LIMIT_MW,'MW','NEA FY2025/26 historical capacity context','2030 scenario'),
    ('Deterministic ENS limit',MAX_ENS_FRACTION,'fraction annual energy',
     'Reliability scenario, not stochastic EENS','Explicit assumption'),
    ('VOLL',VOLL_NPR_per_kWh,'NPR/kWh','JICA Annex 4 quotes earlier MP2014 ENS cost of 0.76 USD/kWh','Historical source proxy, NOT 2030 official VOLL'),
    ('New plant capacity ceilings',str(max_new),'MW','Supplied JICA-labelled expansion scenario',
     'Illustrative upper screening, not commissioning forecast'),
]
provenance=pd.DataFrame(audit_items,columns=['Parameter','Value','Unit','Source_or_method','Evidence_Quality'])
provenance['Source_URL']=''
provenance.loc[provenance['Source_or_method'].str.contains('NEA FY2025/26'), 'Source_URL']='https://www.nea.org.np/en/detail/annual-report-20252026'
provenance.loc[provenance['Source_or_method'].str.contains('JICA'), 'Source_URL']='https://openjicareport.jica.go.jp/pdf/12387999_01.pdf'
provenance.loc[provenance['Source_or_method'].str.contains('Annex 4'), 'Source_URL']='https://openjicareport.jica.go.jp/pdf/12387999_02.pdf'
provenance.to_csv(output_folder/'15_input_evidence_and_assumptions.csv',index=False)
print(provenance[['Parameter','Evidence_Quality']].to_string(index=False))
print('\nOutput folder:',output_folder)

## 21. Interpret the outcome correctly — and the next research upgrades
- **This is a complete runnable LP for one 2030 scenario**, with all 288 representative hourly balances, 12-month resource variation, a daily PROR water balance, a seasonal new-reservoir balance, annualized investment, variable costs, imports, ENS, and two plotted dispatch days.
- **It is not a fully data-verified national optimum.** A material part of existing hydro lacks individually confirmed technology; the 2040 hydro CFs proxy 2030 resources; solar and demand hourly shapes are synthetic; 2030 construction feasibility and tariffs remain uncertain.
- The **zero-new-solar result, if it occurs, is not forced or hidden**. It may follow from CAPEX, CF, import and water assumptions; investigate with a **separate sensitivity** to utility solar procurement prices, PV resource geography and policy requirements. The installed 154.94 MW of solar is present regardless of new build.
- Recommended research upgrades: (1) resolve 1,077.481 MW unknown hydro by actual operating scheme; (2) verify 904.600 MW provisional IPP labels; (3) use national multi-site NASA/PVGIS solar time series; (4) load candidate hydro reservoir volumes/COD and actual CAPEX; (5) improve 2030 import/export prices and limits; (6) incorporate plant outages before calculating *expected* EENS or LOLE; (7) introduce chronological 8,760-hour hydrology/demand variation and transmission bottlenecks.

### Main scholarly references
1. NEA, *A Year in Review, FY2025/26*, Appendix 5, Appendix 6, project facts: https://www.nea.org.np/en/detail/annual-report-20252026
2. JICA/NEA, *Project on Integrated Power System Development Plan in Nepal*, December 2024, Final Report and Annexes: https://openjicareport.jica.go.jp/pdf/12387999_01.pdf and https://openjicareport.jica.go.jp/pdf/12387999_02.pdf and https://openjicareport.jica.go.jp/pdf/12387999_03.pdf
3. NREL, *Cross-Border Energy Trade between Nepal and India: Assessment of Trading Opportunities*, 2019 (short-term pondage and power system trade modeling; modeling approach, **not** a complete verified 2030 input set): https://www.nrel.gov/docs/fy19osti/72066.pdf
4. Supplied files: `nepal_jica_cost_lcoe_parameters.csv`, `nepal_jica_expansion_scenarios.csv`, and your previously generated Step 34 and Steps 36–40 CSVs. **Their derived or manually assembled values are not automatically primary-source validated.**

**Keep this notebook, the data audit CSVs and the results under version control in your existing GEP workspace.** Publish only the modeled results with assumptions and evidence status visible.